In [13]:
from IPython.core.magic import register_cell_magic, register_line_magic
import subprocess
import tempfile
import os
import re
import shutil

_JAVA_COMPILE_DIR = None
_JAVA_CLASSPATH = []


def _get_compile_dir():
    global _JAVA_COMPILE_DIR
    if _JAVA_COMPILE_DIR is None:
        _JAVA_COMPILE_DIR = tempfile.mkdtemp(prefix="java_magic_")
    return _JAVA_COMPILE_DIR


def _cleanup():
    global _JAVA_COMPILE_DIR, _JAVA_CLASSPATH
    if _JAVA_COMPILE_DIR and os.path.exists(_JAVA_COMPILE_DIR):
        shutil.rmtree(_JAVA_COMPILE_DIR, ignore_errors=True)
    _JAVA_COMPILE_DIR = None
    _JAVA_CLASSPATH = []


@register_line_magic
def javacp(line):
    global _JAVA_CLASSPATH
    path = line.strip().strip('"').strip("'")
    if os.path.exists(path):
        _JAVA_CLASSPATH.append(path)
        print("[OK] Added to classpath: " + path)
    else:
        print("[ERR] Not found: " + path)


@register_line_magic
def javacls(line):
    _cleanup()
    print("[OK] Java compile cache cleared")


@register_cell_magic
def java(line, cell):
    opts = []
    parts = line.strip().split()
    while parts and parts[0].startswith('--'):
        opts.append(parts.pop(0))
    
    force_new = '--new' in opts
    show_time = '--time' in opts
    
    if force_new:
        _cleanup()
    
    compile_dir = _get_compile_dir()
    
    public_class = re.search(r'public\s+class\s+(\w+)', cell)
    package_class = re.search(r'class\s+(\w+)', cell)
    
    if public_class:
        classname = public_class.group(1)
    elif package_class:
        classname = package_class.group(1)
    else:
        classname = "Main"
    
    if parts and parts[0][0].isupper() and not parts[0].isdigit():
        cmd_classname = parts.pop(0)
        if cmd_classname != classname and not public_class and not package_class:
            classname = cmd_classname
    else:
        cmd_classname = classname
    
    args = parts
    
    package_match = re.search(r'package\s+([\w.]+);', cell)
    if package_match:
        package_path = package_match.group(1).replace('.', os.sep)
        src_dir = os.path.join(compile_dir, package_path)
        os.makedirs(src_dir, exist_ok=True)
    else:
        src_dir = compile_dir
    
    filepath = os.path.join(src_dir, classname + ".java")
    
    code = cell
    if not re.search(r'class\s+\w+', cell):
        code = (
            "public class " + classname + " {\n"
            "    public static void main(String[] args) {\n"
            "        " + cell.replace('\n', '\n        ') + "\n"
            "    }\n"
            "}"
        )
    
    has_scanner = 'Scanner' in cell and 'System.in' in cell
    input_data = None
    
    if has_scanner and args:
        input_data = '\n'.join(args) + '\n'
        args = []
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(code)
    
    javac_cmd = ['javac', '-encoding', 'UTF-8']
    if _JAVA_CLASSPATH:
        cp = os.pathsep.join(_JAVA_CLASSPATH)
        javac_cmd.extend(['-cp', cp])
    javac_cmd.append(filepath)
    
    compile_result = subprocess.run(
        javac_cmd,
        capture_output=True,
        text=True,
        cwd=compile_dir
    )
    
    if compile_result.returncode != 0:
        errors = compile_result.stderr
        errors = errors.replace(compile_dir + os.sep, '')
        errors = errors.replace(compile_dir, '')
        print("[COMPILE ERROR]\n")
        print(errors)
        return
    
    java_cmd = ['java', '-Dfile.encoding=UTF-8']
    if _JAVA_CLASSPATH:
        cp = os.pathsep.join([compile_dir] + _JAVA_CLASSPATH)
        java_cmd.extend(['-cp', cp])
    else:
        java_cmd.extend(['-cp', compile_dir])
    
    if package_match:
        full_classname = package_match.group(1) + "." + cmd_classname
    else:
        full_classname = cmd_classname
    
    java_cmd.append(full_classname)
    java_cmd.extend(args)
    
    import time
    start = time.time()
    
    try:
        if input_data:
            run_result = subprocess.run(
                java_cmd,
                capture_output=True,
                text=True,
                cwd=compile_dir,
                input=input_data,
                timeout=10
            )
        else:
            run_result = subprocess.run(
                java_cmd,
                capture_output=True,
                text=True,
                cwd=compile_dir,
                timeout=10
            )
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start
        print("[TIMEOUT] Execution timed out after " + str(round(elapsed, 2)) + "s")
        if has_scanner and not input_data:
            print("\n[HINT] Scanner(System.in) detected but no input provided")
            print("       Solutions:")
            print("       1. Hardcode input values")
            print("       2. Pass input as args: %%java ClassName 20")
            print("       3. Use file redirection")
        return
    
    elapsed = time.time() - start
    
    if run_result.stdout:
        print(run_result.stdout, end='')
    
    if run_result.stderr:
        stderr = run_result.stderr
        harmless = ['Picked up JAVA_TOOL_OPTIONS', 'WARNING']
        if not any(h in stderr for h in harmless):
            print("\n[STDERR] " + stderr.strip())
    
    if show_time:
        print("\n[TIME] " + str(round(elapsed, 3)) + "s")


del java, javacp, javacls

数据类型：java是强类型语言，所有数据必须指定类型。\
8种基础数据类型：byte、short、int、long、float、double、char、boolean\
运算符：赋值运算符、算术运算符、比较运算符、逻辑运算符

In [15]:
%%java Test1
public class Test1 {
    public static void main(String[] args){
        //变量  数据类型 变量名 = 变量值
        int id = 1001;
        String name = "jack";
        double score = 95.5;
        boolean isOk = true;

        //赋值运算符 =
        int i = 10;
        double pi = 3.14;

        //算术运算符 + - * / % ++  --
        int  a = 10;
        int b = 3;
        System.out.println(a + " + " + b + "=" + (a+b));//13
        System.out.println(a + " - " + b + "=" + (a-b));//7
        System.out.println(a + " * " + b + "=" + (a*b));//30
        System.out.println(a + " / " + b + "=" + (a/b));//3
        System.out.println(a + " % " + b + "=" + (a%b));//1

        a++;
        System.out.println("a=" + a);//11
        int c = a++;//当a++参与运算，先赋值，后自增
        System.out.println("c=" + c);//11
        System.out.println("a=" + a);//12

        int d = ++a;//当++a参与运算，先自增，后赋值
        System.out.println("d=" + d);//13
        System.out.println("a=" + a);//13

        //比较运算符---结果是boolean
        int num1 = 20;
        int num2 = 10;
        boolean result1 = num1>num2;
        boolean result2 = num1<num2;
        System.out.println(num1 + " > " + num2 + "=" + result1);//true
        System.out.println(num1 + " < " + num2 + "=" + result2);//false

        //逻辑运算符  && || ！
        //&&  当参与运算的每个元素都是true，结果才为true，否则为false
        //|| 当参与运算的每个元素至少有一个true，结果为true，否则为false
        //！ 取反
        boolean result3 = (num1>5)&&(num2<15);
        boolean result4 = (num1>5)||(num2<15);
        boolean result5 = !(num1>5);
        System.out.println(result3);//true
        System.out.println(result4);//true
        System.out.println(result5);//false

        double mathScore = 88.5;
        double englishScore = 95.0;

        double avgScore = (mathScore + englishScore) / 2;
        System.out.println("avgScore=" + avgScore);//avgScore=91.75

        boolean isQualified = avgScore >= 60;
        System.out.println("isQualified=" + isQualified);//

    }
}


10 + 3=13
10 - 3=7
10 * 3=30
10 / 3=3
10 % 3=1
a=11
c=11
a=12
d=13
a=13
20 > 10=true
20 < 10=false
true
true
false
avgScore=91.75
isQualified=true


分支、循环\
分支结构：if、if…else、i…else if..else、switch\
if/else if：区间、范围、多条件组合查询(如分数>60,age>18)\
switch:固定等值匹配（状态码、月份、数字类型标识）


练习：计算两门课程成绩的平均值(double)，判断平均成绩是否大于等于60分。

In [17]:
%%java Test2
public class Test2 {
    public static void main(String[] args) {
        //单分支
        int score = 70;
        if(score>=60){
            System.out.println("及格");
        }

        //双分支
        int age = 17;
        if(age>=18){
            System.out.println("成年");
        }else{
            System.out.println("未成年");
        }

        //多分支
        int score2 = 85;
        if(score2>=90){
            System.out.println("优秀");
        }else if(score>=80){
            System.out.println("良好");
        }else if(score>=70){
            System.out.println("中等");
        }else if(score>=60){
            System.out.println("及格");
        }else{
            System.out.println("不及格");
        }

        //switch
        //status会自上而下匹配，一旦匹配到会停止匹配
        int status = 2;
        switch(status){
            case 1:
                System.out.println("待支付");
                break;
            case 2:
                System.out.println("已支付");
                break;
            case 3:
                System.out.println("已发货");
                break;
            default:
                System.out.println("无效状态");
                break;
        }
        // 后续代码
    }
}


及格
未成年
中等
已支付


循环结构：for、while、do…while\
break、continue

In [18]:
%%java Test3
public class Test3 {
    public static void main(String[] args) {
        //for
        for(int i=1;i<=5;i++){
            System.out.println("第" + i + "条数据");//第1 2 3 4 5条数据
        }

        //while--游标
        //模拟游标读取数据，直到无记录
        int count = 1;
        while(count <= 3){
            System.out.println("读取第" + count + "行");//
            count++;//手动自增，避免无限循环
        }

        //break -- 跳出当前层整个循环
        int j=1;
        for(;j<=10;j++){
            if(j==5){
                System.out.println("终止整个本层循环");
                break;
            }
            System.out.println("j=" + j);//1 2 3 4
        }

        //continue -- 跳出当前层本次循环，继续下一次循环
        int k=1;
        for(;k<=10;k++){
            if(k==5){
                System.out.println("终止本层循环的本次循环，后续循环继续");
                continue;
            }
            System.out.println("k=" + k);//
        }
    }
}


第1条数据
第2条数据
第3条数据
第4条数据
第5条数据
读取第1行
读取第2行
读取第3行
j=1
j=2
j=3
j=4
终止整个本层循环
k=1
k=2
k=3
k=4
终止本层循环的本次循环，后续循环继续
k=6
k=7
k=8
k=9
k=10


练习:

1、输入一个整数n（1-100），遍历n\
跳过所有偶数；\
判断奇数是否为素数\
分别打印：奇数素数、奇数非素数

In [20]:
%%java Test4 5
import java.util.Scanner;
public class Test4 {
public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);
        //提示语
        System.out.println("请输入一个整数：");
        int n = scanner.nextInt();

        for(int i=1; i<=n; i++){
            if(i%2 == 0){
                continue;
            }
            if(i == 1){
                System.out.println(i +"是奇数，不是素数");
                continue;
            }

            //判断素数
            boolean isPrime = true;//默认是素数
            for(int j=2; j<i; j++){
                if(i%j == 0){
                    isPrime = false;
                    break;
                }
            }

            if(isPrime){
                System.out.println(i +"是奇数，也是素数");
            }else{
                System.out.println(i +"是奇数，但不是素数");
            }
        }
    }
}

请输入一个整数：
1是奇数，不是素数
3是奇数，也是素数
5是奇数，也是素数


银行系统密码校验

设定正确密码123456，用户最多输入3次\
输入错误提示剩余次数，继续循环\
输入正确直接break，提示登录成功\
3次全部错误，锁定账户

In [21]:
%%java PasswordCheck 123456

import java.util.Scanner;

public class PasswordCheck {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);

        int password = 123456;

        for(int i=1; i<=3; i++){
            System.out.println("请输入密码：");
            int pwd = scanner.nextInt();
            if(password != pwd){
                int left = 3 - i;
                if(left == 0){
                    System.out.println("密码错误3次，账户锁定");
                }else{
                    System.out.println("密码错误，还剩" + left + "次机会");
                }
                continue;
            }
            System.out.println("密码正确，欢迎进入系统");
            break;
        }
    }
}

请输入密码：
密码正确，欢迎进入系统


In [22]:
%%java PasswordCheck2 111111 222222 333333

import java.util.Scanner;

public class PasswordCheck2 {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);

        int password = 123456;

        for(int i=1; i<=3; i++){
            System.out.println("请输入密码：");
            int pwd = scanner.nextInt();
            if(password != pwd){
                int left = 3 - i;
                if(left == 0){
                    System.out.println("密码错误3次，账户锁定");
                }else{
                    System.out.println("密码错误，还剩" + left + "次机会");
                }
                continue;
            }
            System.out.println("密码正确，欢迎进入系统");
            break;
        }
    }
}

请输入密码：
密码错误，还剩2次机会
请输入密码：
密码错误，还剩1次机会
请输入密码：
密码错误3次，账户锁定


输入年份、月份，判断当月天数

1/3/5/7/8/10/1月31天\
4/6/9/11月30天\
2月28/29 判断闰年

In [23]:
%%java Calendar 2024 2 2023 5 0

import java.util.Scanner;

public class Calendar {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);

        while(true){
            System.out.println("输入年份(当输入0退出)：");
            int year = scanner.nextInt();
            if(year == 0){
                System.out.println("退出程序");
                break;
            }

            System.out.println("请输入月份:");
            int month = scanner.nextInt();
            int day = 0;
            switch(month){
                case 1:
                case 3:
                case 5:
                case 7:
                case 8:
                case 10:
                case 12:
                    day = 31;
                    break;
                case 4:
                case 6:
                case 9:
                case 11:
                    day=30;
                    break;
                case 2:
                    if(year % 4 == 0 && year % 100 != 0 || year % 400 == 0){
                        day = 29;
                    }else{
                        day = 28;
                    }
                    break;
            }
            System.out.println(year + "年" + month + "月有" + day + "天");
        }
    }
}

输入年份(当输入0退出)：
请输入月份:
2024年2月有29天
输入年份(当输入0退出)：
请输入月份:
2023年5月有31天
输入年份(当输入0退出)：
退出程序


数组、集合\
数组：相同数据类型数据的容器，长度固定，创建后不能改变长度。\
存储有序，通过下标访问/赋值，下标从0开始  数组名\[下标]\
特点：长度固定，增删元素效率较低，查询速度快\

In [24]:
%%java Test7

public class Test7 {
    public static void main(String[] args) {
        int[] arr1;
        arr1 = new int[3];

        arr1[0] = 10;
        arr1[1] = 20;
        arr1[2] = 30;

        double[] arr2 = new double[2];
        arr2[0] = 1.1;
        arr2[1] = 2.2;

        String[] arr3 = {"张三", "李四", "王五"};

        System.out.println("arr1的长度是：" + arr1.length);

        for(int i=0; i<arr1.length; i++){
            System.out.println("arr1[" + i + "] = " + arr1[i]);
        }

        for (double element : arr2){
            System.out.println("姓名 = " + element);
        }
    }
}

arr1的长度是：3
arr1[0] = 10
arr1[1] = 20
arr1[2] = 30
姓名 = 1.1
姓名 = 2.2


练习1：定义数组{12,45,7,89,23},遍历求和并找出最大值

In [25]:
%%java Test8

public class Test8 {
    public static void main(String[] args) {
        int[] arr = {12,45,7,89,23};
        int sum = 0;
        int max = arr[0];

        for (int i = 0; i < arr.length; i++){
            sum += arr[i];
            if (arr[i] > max){
                max = arr[i];
            }
        }

        System.out.println("数组总和:" + sum);
        System.out.println("数组的最大值:" + max);
    }
}

数组总和:176
数组的最大值:89


练习2：定义数组{5,11,33,77,99}，输入数字，输出该数字对应的下标，不存在提示-1

In [26]:
%%java Test9 33

import java.util.Scanner;

public class Test9 {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);
        int[] arr = {5, 11, 33, 77, 99};

        System.out.println("请输入要查找的数字:");
        int target = scanner.nextInt();
        int index = -1;

        for (int i = 0; i < arr.length; i++) {
            if (arr[i] == target) {
                index = i;
                break;
            }
        }
        System.out.println("目标元素的下标:" + index);
    }
}

请输入要查找的数字:
目标元素的下标:2


In [27]:
%%java Test9a 100

import java.util.Scanner;

public class Test9a {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);
        int[] arr = {5, 11, 33, 77, 99};

        System.out.println("请输入要查找的数字:");
        int target = scanner.nextInt();
        int index = -1;

        for (int i = 0; i < arr.length; i++) {
            if (arr[i] == target) {
                index = i;
                break;
            }
        }
        System.out.println("目标元素的下标:" + index);
    }
}

请输入要查找的数字:
目标元素的下标:-1


数组总结：

长度固定，不能动态增减\
删除/插入元素需要手动移动大量数据，效率比较低\
开发中频繁增删数据优先使用集合。

------

集合：

长度可变的容器，只能存储引用类型（存基本类型自动装箱）\

分为两大体系：\
Collection单列集合：List、Set\
Map双列集合:一次存一对键值对key-value\
优点：自动扩容，提供大量增删改查工具方法，开发最常用\

List—有序、可重复、有索引\
ArrayList---底层数组

In [28]:
%%java Test10

import java.util.ArrayList;

public class Test10 {
    public static void main(String[] args) {
        ArrayList<String> list = new ArrayList<>();

        list.add("张三");
        list.add("王五");
        list.add("王五");
        list.add("赵六");
        list.add("孙七");

        System.out.println("集合的长度:" + list.size());

        System.out.println("获取索引为2的元素:" + list.get(2));

        list.set(1, "李四四");

        list.remove(3);

        for (int i = 0; i < list.size(); i++) {
            System.out.println("list[" + i + "] = " + list.get(i));
        }
        System.out.println("***************************");
        for(String element : list){
            System.out.println("element = " + element);
        }
    }
}

集合的长度:5
获取索引为2的元素:王五
list[0] = 张三
list[1] = 李四四
list[2] = 王五
list[3] = 孙七
***************************
element = 张三
element = 李四四
element = 王五
element = 孙七
